In [1]:
import os
import time

from bng_simulator.utils.services_utils import send_request
from bng_simulator.utils.io_dict_utils import(
    save_yaml,
    round_dict_values,
    build_tree_from_dict,
    convert_tree_into_proper_dict
)

In [2]:
# Get config, scenario , infos
scenario_infos = send_request("get_sim_config")

In [3]:
# Print the vehicles in the environment
for v, v_info in scenario_infos["vehicles"].items():
    _infos = {_k : _v for _k, _v in v_info.items() if _k not in ["scenario_args", "sensors"]}
    print(f"Vehicle {v} :")
    for _k, _v in _infos.items():
        print(f"\t{_k} : {_v}")

Vehicle ego :
	color : blue
	license : EGO
	model : bolide


In [7]:
# Print the part numbers, used as unique identifiers for the vehicle components
for v_name, v_part in scenario_infos["vehicles_part"].items():
    print(f"Vehicle {v_name} :  {v_part}")

Vehicle ego :  bolide_350


In [8]:
# Let's enumerate all gear options and their characteristics
def extract_gear_infos(vehicle_name = None):
    """
    Go through all gears and extract their characteristics
    """
    in_args = {}
    in_args_set = {}
    if vehicle_name is not None:
        in_args["vehicle_name"] = vehicle_name
        in_args_set["vehicle_name"] = vehicle_name
    # Let's first get current infos
    gear_infos = send_request("get_gearbox_info", in_args)
    min_gear = int(gear_infos["minGearIndex"])
    max_gear = int(gear_infos["maxGearIndex"])
    # TODO: The min is actually negative of the actual min cause of abs.
    print(f"Min/max gear: {min_gear}/{max_gear}")
    
    # A function to parse the gear infos
    def parse_gear_infos(gear_infos):
        curr_gear = int(gear_infos["gearIndex"])
        gear_mode_index = gear_infos["gearModeIndex"]
        gear_ratio = gear_infos["gearRatio"]
        gear_name = gear_infos["gearName"]
        gear_mode = gear_infos["mode"]
        return gear_name, \
            {"index": curr_gear, "mode_index": gear_mode_index, 
             "gear_ratio": gear_ratio, "mode": gear_mode}
    
    # Store the different gear ratio
    mapping_info = {}
    gear_name, gear_info = parse_gear_infos(gear_infos)
    print(f"Gear {gear_name} - {gear_info}")
    initial_gear_name = gear_name
    for i in range(-1, max_gear + 3): # Just extra for the sake of it
        in_args_set["gear_index"] = i
        send_request("set_gearbox_index", in_args_set)
        # Wait for a bit
        time.sleep(0.5)
        # Get the gear infos
        gear_infos = send_request("get_gearbox_info", in_args)
        gear_name, gear_info = parse_gear_infos(gear_infos)
        gear_info["indexCmd"] = i
        print(f"Gear {gear_name} - {gear_info}")
        if gear_name not in mapping_info:
            mapping_info[gear_name] = gear_info
    # Set the initial gear index
    initial_gear_index = mapping_info[initial_gear_name]["indexCmd"]
    in_args_set["gear_index"] = initial_gear_index
    send_request("set_gearbox_index", in_args_set)
    return mapping_info

In [9]:
# Extract the gear infos
gear_infos = extract_gear_infos()

Min/max gear: -1/5
Gear -1.0 - {'index': -1, 'mode_index': 0.0, 'gear_ratio': -3.14, 'mode': ''}
Gear -1.0 - {'index': -1, 'mode_index': 0.0, 'gear_ratio': -3.14, 'mode': '', 'indexCmd': -1}
Gear 0.0 - {'index': 0, 'mode_index': 0.0, 'gear_ratio': 0.0, 'mode': '', 'indexCmd': 0}
Gear 1.0 - {'index': 1, 'mode_index': 0.0, 'gear_ratio': 3.305, 'mode': '', 'indexCmd': 1}
Gear 2.0 - {'index': 2, 'mode_index': 0.0, 'gear_ratio': 2.27, 'mode': '', 'indexCmd': 2}
Gear 3.0 - {'index': 3, 'mode_index': 0.0, 'gear_ratio': 1.6300000000000001, 'mode': '', 'indexCmd': 3}
Gear 4.0 - {'index': 4, 'mode_index': 0.0, 'gear_ratio': 1.2, 'mode': '', 'indexCmd': 4}
Gear 5.0 - {'index': 5, 'mode_index': 0.0, 'gear_ratio': 0.8800000000000001, 'mode': '', 'indexCmd': 5}
Gear 5.0 - {'index': 5, 'mode_index': 0.0, 'gear_ratio': 0.8800000000000001, 'mode': '', 'indexCmd': 6}
Gear 5.0 - {'index': 5, 'mode_index': 0.0, 'gear_ratio': 0.8800000000000001, 'mode': '', 'indexCmd': 7}


In [10]:
gear_infos

{-1.0: {'index': -1,
  'mode_index': 0.0,
  'gear_ratio': -3.14,
  'mode': '',
  'indexCmd': -1},
 0.0: {'index': 0,
  'mode_index': 0.0,
  'gear_ratio': 0.0,
  'mode': '',
  'indexCmd': 0},
 1.0: {'index': 1,
  'mode_index': 0.0,
  'gear_ratio': 3.305,
  'mode': '',
  'indexCmd': 1},
 2.0: {'index': 2,
  'mode_index': 0.0,
  'gear_ratio': 2.27,
  'mode': '',
  'indexCmd': 2},
 3.0: {'index': 3,
  'mode_index': 0.0,
  'gear_ratio': 1.6300000000000001,
  'mode': '',
  'indexCmd': 3},
 4.0: {'index': 4,
  'mode_index': 0.0,
  'gear_ratio': 1.2,
  'mode': '',
  'indexCmd': 4},
 5.0: {'index': 5,
  'mode_index': 0.0,
  'gear_ratio': 0.8800000000000001,
  'mode': '',
  'indexCmd': 5}}

In [11]:
# Useful functions to format the powertrain structure
def format_powertrain_structure(data, relevant_keys):
    """
    Format the powertrain structure.
    
    Args:
        data (Dict[str, Any]): The data.
        relevant_keys (Tuple[str]): The relevant keys.
        
    Returns:
        Dict[str, Any]: The tree-like structure.
    """
    roots, tree = build_tree_from_dict(data)
    res = {}
    for root in roots:
        res[root] = convert_tree_into_proper_dict(root, tree, data, relevant_keys)
    # # Let;s sort it
    # res = dict
    return round_dict_values(res)

powertrain_prop = send_request("get_powertrain_properties")
powertrain_infos = format_powertrain_structure(
    powertrain_prop,
    ['type', 'gearRatio', 'gearRatios', 'availableModes', 'diffTorqueSplit']
)

In [12]:
powertrain_infos

{'mainEngine': {'type': 'combustionEngine',
  'clutch': {'type': 'frictionClutch',
   'gearRatio': 1.0,
   'gearbox': {'type': 'manualGearbox',
    'gearRatio': -3.14,
    'gearRatios': {-1.0: -3.14,
     0.0: 0.0,
     1.0: 3.305,
     2.0: 2.27,
     3.0: 1.63,
     4.0: 1.2,
     5.0: 0.88},
    'differential_R': {'type': 'differential',
     'gearRatio': 3.45,
     'availableModes': ['lsd'],
     'diffTorqueSplit': 0.5,
     'wheelaxleRL': {'type': 'shaft',
      'gearRatio': 1.0,
      'availableModes': ['connected'],
      'spindleRL': {'type': 'shaft',
       'gearRatio': 1.0,
       'availableModes': ['connected']}},
     'wheelaxleRR': {'type': 'shaft',
      'gearRatio': 1.0,
      'availableModes': ['connected'],
      'spindleRR': {'type': 'shaft',
       'gearRatio': 1.0,
       'availableModes': ['connected']}}}}}},
 'wheelaxleFL': {'type': 'shaft',
  'gearRatio': 1.0,
  'availableModes': ['connected']},
 'wheelaxleFR': {'type': 'shaft',
  'gearRatio': 1.0,
  'availableMo

In [13]:
# Get controllers infos
controllers_infos = send_request("get_controller_infos")
controllers_infos = dict(sorted(controllers_infos.items()))

In [14]:
controllers_infos

{'analogOdometer': 'gauges/analogOdometer',
 'doorLCoupler': 'advancedCouplerControl',
 'doorRCoupler': 'advancedCouplerControl',
 'gtState0': 'xlab/gtState',
 'hPattern': 'propAnimation/hPattern',
 'hoodCatchCoupler': 'advancedCouplerControl',
 'hoodLatchCoupler': 'advancedCouplerControl',
 'tailgateCatch': 'advancedCouplerControl',
 'tailgateCoupler': 'advancedCouplerControl',
 'vehicleController': 'vehicleController/vehicleController'}

In [15]:
# Kinematics infos
kin_args = {
    "world_space": False
}
kin_props = send_request("get_vehicle_properties", kin_args)

In [16]:
# Handle engine infos
engine_infos = send_request("get_engine_infos")

In [17]:
engine_infos

{'fuelCapacity': 50.0,
 'fuelVolume': 49.86852384144912,
 'idleRPM': 1200.0,
 'maxRPM': 7500.0,
 'superchargerBoostMax': -1.0,
 'turboBoostMax': -1.0}

In [18]:
# Let's try to output all these infos in a yaml file
vehicle_config_folder = "../config/vehicles/"
veh_name = "ego"
vehicle_path = vehicle_config_folder + scenario_infos["vehicles_part"][veh_name] + ".yaml"

final_dict_params = {
    **round_dict_values(kin_props),
    "engine" : round_dict_values(engine_infos),
    "gearbox": gear_infos,
    "powertrain": round_dict_values(powertrain_infos),
    "controllers": controllers_infos,
}
save_yaml(
    final_dict_params,
    vehicle_path,
    sort_keys=False
)